# Lab 5: Behavioral Cloning - Offline Training

Train a neural network to imitate human driving from recorded state-action pairs.

**Pipeline:**
1. Load recorded data (`.pkl` files from `data_recorder_node`)
2. Preprocess: extract features, normalize
3. Train/val split
4. Train PyTorch MLP
5. Evaluate and visualize
6. Save model for deployment with `bc_eval_node`

In [ ]:
import sys
import os
import pickle
import glob
import numpy as np
import matplotlib.pyplot as plt
import torch

# Add the Lab5 scripts to path so we can import the network
lab5_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, os.path.join(lab5_dir, "scripts"))

from neural_network import BCNetwork

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 1. Load Data

Load one or more `.pkl` files from the `data/` directory. Each file contains:
- `states`: (N, 6) array — [x, y, yaw, v_long, v_lat, omega]
- `actions`: (N, 2) array — [speed, steering_angle]
- `timestamps`: (N,) array

In [ ]:
data_dir = os.path.join(lab5_dir, "data")

# Load all .pkl files and concatenate
pkl_files = sorted(glob.glob(os.path.join(data_dir, "bc_data_*.pkl")))
print(f"Found {len(pkl_files)} data files:")
for f in pkl_files:
    print(f"  {os.path.basename(f)}")

all_states = []
all_actions = []

for f in pkl_files:
    with open(f, "rb") as fp:
        data = pickle.load(fp)
    all_states.append(data["states"])
    all_actions.append(data["actions"])
    print(f"  {os.path.basename(f)}: {data['states'].shape[0]} samples")

states = np.concatenate(all_states, axis=0)
actions = np.concatenate(all_actions, axis=0)

print(f"\nTotal: {states.shape[0]} samples")
print(f"State labels:  {data['state_labels']}")
print(f"Action labels: {data['action_labels']}")

## 2. Explore Data

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 6))

state_labels = ["x", "y", "yaw", "v_long", "v_lat", "omega"]
action_labels = ["speed", "steering_angle"]

# Plot state distributions
for i, label in enumerate(state_labels):
    ax = axes[0, i] if i < 4 else axes[1, i - 4]
    ax.hist(states[:, i], bins=50, alpha=0.7)
    ax.set_title(label)
    ax.set_xlabel("Value")

# Plot action distributions
for i, label in enumerate(action_labels):
    ax = axes[1, len(state_labels) - 4 + i]
    ax.hist(actions[:, i], bins=50, alpha=0.7, color="orange")
    ax.set_title(label)
    ax.set_xlabel("Value")

plt.tight_layout()
plt.show()

# Plot trajectory
fig, ax = plt.subplots(1, 1, figsize=(8, 8))
scatter = ax.scatter(states[:, 0], states[:, 1], c=np.arange(len(states)), cmap="viridis", s=1)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Recorded Trajectory")
ax.set_aspect("equal")
plt.colorbar(scatter, label="Time step")
plt.show()

## 3. Prepare Features

Select which state features to use as model input. Default: all 6 state features `[x, y, yaw, v_long, v_lat, omega]`.

You can experiment with different feature subsets by changing `feature_indices` below (e.g., `[0, 1, 2]` for position + heading only).

In [ ]:
# Feature selection
# State columns: [0:x, 1:y, 2:yaw, 3:v_long, 4:v_lat, 5:omega]
feature_indices = [0, 1, 2, 3, 4, 5]  # all state features
feature_names = [state_labels[i] for i in feature_indices]
print(f"Using features: {feature_names}")

X = states[:, feature_indices].copy()
Y = actions.copy()

# Normalize features (zero mean, unit variance)
X_mean = X.mean(axis=0)
X_std = X.std(axis=0)
X_std[X_std < 1e-8] = 1.0  # avoid division by zero

X_norm = (X - X_mean) / X_std

# Normalize actions too
Y_mean = Y.mean(axis=0)
Y_std = Y.std(axis=0)
Y_std[Y_std < 1e-8] = 1.0

Y_norm = (Y - Y_mean) / Y_std

print(f"X shape: {X_norm.shape}, Y shape: {Y_norm.shape}")
print(f"X_mean: {X_mean}, X_std: {X_std}")
print(f"Y_mean: {Y_mean}, Y_std: {Y_std}")

## 4. Train/Validation Split

In [ ]:
val_ratio = 0.15
n = X_norm.shape[0]

# Compute per-sample weights based on steering magnitude
# Higher |steering| = higher weight so the model focuses on turns
steer_magnitude = np.abs(Y[:, 1])  # unnormalized steering
turn_weight = 5.0  # how much more to weight turns vs straight
weights = 1.0 + turn_weight * (steer_magnitude / (steer_magnitude.max() + 1e-8))
weights = weights / weights.mean()  # normalize so mean weight = 1

print(f"Weight range: {weights.min():.2f} - {weights.max():.2f}")
print(f"Samples with weight > 2: {(weights > 2).sum()} / {n}")

idx = np.random.permutation(n)
n_val = int(n * val_ratio)

X_val, Y_val = X_norm[idx[:n_val]], Y_norm[idx[:n_val]]
X_train, Y_train = X_norm[idx[n_val:]], Y_norm[idx[n_val:]]
W_train = weights[idx[n_val:]]

print(f"Train: {X_train.shape[0]} samples")
print(f"Val:   {X_val.shape[0]} samples")

## 5. Train Model

In [ ]:
# Hyperparameters
input_size = len(feature_indices)
output_size = 2  # [speed, steering_angle]
hidden_sizes = [64, 128, 128, 64]
learning_rate = 1e-3
n_epochs = 75
batch_size = 64

model = BCNetwork(
    input_size=input_size,
    output_size=output_size,
    hidden_sizes=hidden_sizes,
    lr=learning_rate,
)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
train_losses = []
val_losses = []

for epoch in range(n_epochs):
    train_loss = model.train_epoch(X_train, Y_train, w=W_train, batch_size=batch_size)
    train_losses.append(train_loss)

    # Validation loss (unweighted — true performance)
    model.eval()
    with torch.no_grad():
        val_pred = model.predict(X_val)
        val_loss = np.mean((val_pred - Y_val) ** 2)
    val_losses.append(val_loss)

    if (epoch + 1) % 20 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:4d}/{n_epochs}  train_loss={train_loss:.6f}  val_loss={val_loss:.6f}")

## 6. Visualize Results

In [ ]:
# Loss curves
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.plot(train_losses, label="Train Loss")
ax.plot(val_losses, label="Val Loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.set_title("Training Progress")
ax.legend()
ax.set_yscale("log")
ax.grid(True)
plt.show()

# Prediction vs ground truth on validation set
val_pred = model.predict(X_val)

# Denormalize
val_pred_denorm = val_pred * Y_std + Y_mean
Y_val_denorm = Y_val * Y_std + Y_mean

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, label in enumerate(action_labels):
    ax = axes[i]
    ax.scatter(Y_val_denorm[:, i], val_pred_denorm[:, i], s=1, alpha=0.3)
    lims = [
        min(Y_val_denorm[:, i].min(), val_pred_denorm[:, i].min()),
        max(Y_val_denorm[:, i].max(), val_pred_denorm[:, i].max()),
    ]
    ax.plot(lims, lims, "r--", linewidth=1)
    ax.set_xlabel(f"True {label}")
    ax.set_ylabel(f"Predicted {label}")
    ax.set_title(f"{label}: Predicted vs True")
    ax.set_aspect("equal")
    ax.grid(True)

plt.tight_layout()
plt.show()

## 7. Save Model and Normalization Stats

Save the model `.pt` file and normalization parameters to the `models/` directory.
The eval node needs both to run inference.

In [ ]:
model_dir = os.path.join(lab5_dir, "models")
os.makedirs(model_dir, exist_ok=True)

# Save model weights
model_path = os.path.join(model_dir, "bc_model.pt")
model.save_model(model_path)
print(f"Model saved to {model_path}")

# Save normalization stats (needed at inference time)
norm_path = os.path.join(model_dir, "normalization.pkl")
norm_stats = {
    "X_mean": X_mean,
    "X_std": X_std,
    "Y_mean": Y_mean,
    "Y_std": Y_std,
    "feature_indices": feature_indices,
    "feature_names": feature_names,
    "input_size": input_size,
    "output_size": output_size,
    "hidden_sizes": hidden_sizes,
}
with open(norm_path, "wb") as f:
    pickle.dump(norm_stats, f)
print(f"Normalization stats saved to {norm_path}")

print("\nTo deploy, update lab5_eval.yaml with:")
print(f"  model_path: {model_path}")
print(f"  norm_path:  {norm_path}")